# Daily Challenge: Preprocess & Fine-tune Transformer Models
## Week 7 — Day 3 | DI GenAI & Machine Learning Bootcamp 2026

**Objectives:**
- Understand how BERT and XLM-RoBERTa process text through tokenization
- Tokenize text using `encode_plus` and explore `input_ids`, `attention_mask`, and labels
- Prepare model-ready input tensors with proper padding, truncation, and special tokens
- Load and explore a text classification dataset from CSV files
- Structure training data using k-fold cross-validation with `StratifiedKFold`

**Parts:**
1. Understanding BERT and XLM-RoBERTa
2. Tokenizing Text
3. Preparing Input Data for the Model
4. Loading and Exploring the Dataset
5. Creating Cross-Validation Folds

> **Run on Google Colab** — GPU recommended for fine-tuning.

In [ ]:
!pip install --quiet transformers torch datasets scikit-learn pandas matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from collections import Counter
from transformers import BertTokenizer, XLMRobertaTokenizer
from sklearn.model_selection import StratifiedKFold
import warnings
warnings.filterwarnings("ignore")

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch  {torch.__version__} ✓")
print(f"Device   : {device}")

## Part 1 — Understanding BERT and XLM-RoBERTa

### BERT (Bidirectional Encoder Representations from Transformers)

| Property | Detail |
|---|---|
| **Architecture** | Encoder-only Transformer (bidirectional self-attention) |
| **Pretraining** | Masked Language Modeling (MLM) + Next Sentence Prediction (NSP) |
| **Tokenizer** | WordPiece — 30,522-token vocabulary |
| **Special tokens** | `[CLS]` (classification) · `[SEP]` (separator) · `[PAD]` (padding) · `[MASK]` (MLM) |
| **Input format** | `[CLS] sentence_A [SEP] sentence_B [SEP]` |
| **Common variants** | `bert-base-uncased`, `bert-base-cased`, `bert-large-uncased`, `bert-base-multilingual-cased` |
| **Strength** | Rich bidirectional context; strong on NLU benchmarks (GLUE, SQuAD) |

### XLM-RoBERTa (Cross-lingual Language Model — RoBERTa)

| Property | Detail |
|---|---|
| **Architecture** | Encoder-only Transformer (RoBERTa-style — removes NSP, trains longer on more data) |
| **Pretraining** | MLM only on 100+ languages from CommonCrawl (2.5 TB of text) |
| **Tokenizer** | SentencePiece (Unigram LM) — 250,002-token multilingual vocabulary |
| **Special tokens** | `<s>` (start) · `</s>` (end/separator) · `<pad>` (padding) · `<mask>` (MLM) |
| **Input format** | `<s> sentence_A </s></s> sentence_B </s>` |
| **Common variants** | `xlm-roberta-base` (270M params), `xlm-roberta-large` (560M params) |
| **Strength** | Excellent cross-lingual transfer; one model for 100+ languages |

### Key Difference for Tokenization

```
Text: "Fine-tuning transformers"

BERT   (WordPiece) → [CLS] fine - ##tuning transformers [SEP]
XLM-R  (SentencePiece) → <s> ▁Fine - tuning ▁transformers </s>
```

The `▁` (underscore) prefix in SentencePiece marks the start of a new word (equivalent to BERT's `##` logic, but inverted).

In [ ]:
# Load both tokenizers
bert_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
xlmr_tokenizer = XLMRobertaTokenizer.from_pretrained("xlm-roberta-base")

print("Tokenizers loaded ✓")
print(f"\nBERT   vocab size : {bert_tokenizer.vocab_size:,}")
print(f"XLM-R  vocab size : {xlmr_tokenizer.vocab_size:,}")

# Special tokens comparison
print(f"\n{'─'*50}")
print(f"{'Token role':<20} {'BERT':>12}  {'XLM-RoBERTa':>15}")
print(f"{'─'*50}")
roles = [
    ("CLS / Start",  bert_tokenizer.cls_token,  xlmr_tokenizer.bos_token),
    ("SEP / End",    bert_tokenizer.sep_token,  xlmr_tokenizer.eos_token),
    ("PAD",          bert_tokenizer.pad_token,  xlmr_tokenizer.pad_token),
    ("MASK",         bert_tokenizer.mask_token, xlmr_tokenizer.mask_token),
    ("UNK",          bert_tokenizer.unk_token,  xlmr_tokenizer.unk_token),
]
for role, b_tok, x_tok in roles:
    print(f"{role:<20} {str(b_tok):>12}  {str(x_tok):>15}")
print(f"{'─'*50}")

## Part 2 — Tokenizing Text

`encode_plus` is the core method for preparing single sequences or sentence pairs. It returns a dictionary with:
- **`input_ids`** — integer IDs for each token (including special tokens)
- **`attention_mask`** — 1 for real tokens, 0 for padding tokens
- **`token_type_ids`** (BERT only) — 0 for sentence A tokens, 1 for sentence B tokens

In [ ]:
sentence = "Fine-tuning transformers is essential for NLP tasks."

MAX_LEN = 20

# ── BERT single-sentence ──────────────────────────────────────────────────────
bert_enc = bert_tokenizer.encode_plus(
    sentence,
    add_special_tokens = True,
    max_length         = MAX_LEN,
    padding            = "max_length",
    truncation         = True,
    return_attention_mask   = True,
    return_token_type_ids   = True,
)

bert_tokens = bert_tokenizer.convert_ids_to_tokens(bert_enc["input_ids"])

print("BERT — Single sentence encode_plus")
print(f"  Input  : \"{sentence}\"")
print(f"\n  {'Token':<18} {'ID':>6}  {'Attn':>6}  {'TypeID':>6}")
print(f"  {'─'*44}")
for tok, tid, am, tt in zip(bert_tokens,
                             bert_enc["input_ids"],
                             bert_enc["attention_mask"],
                             bert_enc["token_type_ids"]):
    note = ""
    if tok in ("[CLS]", "[SEP]"): note = f"  ← special"
    elif tok == "[PAD]":           note = f"  ← padding"
    elif tok.startswith("##"):     note = f"  ← sub-word"
    print(f"  {tok:<18} {tid:>6}  {am:>6}  {tt:>6}{note}")

# Decode back to text
decoded = bert_tokenizer.decode(bert_enc["input_ids"], skip_special_tokens=True)
print(f"\n  Decoded (skip_special_tokens=True) : '{decoded}'")

In [ ]:
# ── BERT two-sentence pair ────────────────────────────────────────────────────
sent_a = "BERT reads text bidirectionally."
sent_b = "GPT generates text left to right."

bert_pair = bert_tokenizer.encode_plus(
    sent_a, sent_b,
    add_special_tokens      = True,
    max_length              = 24,
    padding                 = "max_length",
    truncation              = True,
    return_attention_mask   = True,
    return_token_type_ids   = True,
)

pair_tokens = bert_tokenizer.convert_ids_to_tokens(bert_pair["input_ids"])
print("BERT — Two-sentence pair (token_type_ids show A vs B)")
print(f"  Sentence A : \"{sent_a}\"")
print(f"  Sentence B : \"{sent_b}\"")
print(f"\n  {'Token':<16} {'ID':>6}  {'Attn':>6}  {'TypeID':>6}  Segment")
print(f"  {'─'*52}")
for tok, tid, am, tt in zip(pair_tokens,
                              bert_pair["input_ids"],
                              bert_pair["attention_mask"],
                              bert_pair["token_type_ids"]):
    seg = "A" if tt == 0 else "B"
    flag = " ← padding" if tok == "[PAD]" else ""
    print(f"  {tok:<16} {tid:>6}  {am:>6}  {tt:>6}  [{seg}]{flag}")

# ── XLM-RoBERTa single-sentence ───────────────────────────────────────────────
print(f"\n{'─'*55}")
xlmr_enc = xlmr_tokenizer.encode_plus(
    sentence,
    add_special_tokens      = True,
    max_length              = MAX_LEN,
    padding                 = "max_length",
    truncation              = True,
    return_attention_mask   = True,
)

xlmr_tokens = xlmr_tokenizer.convert_ids_to_tokens(xlmr_enc["input_ids"])
print("XLM-RoBERTa — Single sentence encode_plus")
print(f"  Input  : \"{sentence}\"")
print(f"\n  {'Token':<18} {'ID':>8}  {'Attn':>6}")
print(f"  {'─'*38}")
for tok, tid, am in zip(xlmr_tokens, xlmr_enc["input_ids"], xlmr_enc["attention_mask"]):
    note = ""
    if tok in ("<s>", "</s>"): note = "  ← special"
    elif tok == "<pad>":        note = "  ← padding"
    elif "▁" in tok:            note = "  ← word-start marker"
    print(f"  {tok:<18} {tid:>8}  {am:>6}{note}")

decoded_xlmr = xlmr_tokenizer.decode(xlmr_enc["input_ids"], skip_special_tokens=True)
print(f"\n  Decoded : '{decoded_xlmr}'")

## Part 3 — Preparing Input Data for the Model

In [ ]:
# ── Inspect special tokens maps ───────────────────────────────────────────────
print("BERT special_tokens_map:")
for role, tok in bert_tokenizer.special_tokens_map.items():
    print(f"  {role:<20} : {tok}")

print(f"\nXLM-RoBERTa special_tokens_map:")
for role, tok in xlmr_tokenizer.special_tokens_map.items():
    print(f"  {role:<20} : {tok}")

print(f"\nBERT   vocab_size : {bert_tokenizer.vocab_size:,}")
print(f"XLM-R  vocab_size : {xlmr_tokenizer.vocab_size:,}")

In [ ]:
def prepare_bert_input(texts: list[str], tokenizer, max_length: int = 128,
                       labels: list[int] = None) -> dict:
    """
    Batch-tokenize texts with padding + truncation and return tensors
    ready to pass directly to a BERT model.

    Returns a dict with input_ids, attention_mask, (token_type_ids), (labels).
    """
    encoding = tokenizer(
        texts,
        add_special_tokens    = True,
        max_length            = max_length,
        padding               = "max_length",
        truncation            = True,
        return_attention_mask = True,
        return_tensors        = "pt",
    )
    if labels is not None:
        encoding["labels"] = torch.tensor(labels, dtype=torch.long)
    return encoding


def prepare_xlmr_input(texts: list[str], tokenizer, max_length: int = 128,
                        labels: list[int] = None) -> dict:
    """Same as above but for XLM-RoBERTa (no token_type_ids)."""
    encoding = tokenizer(
        texts,
        add_special_tokens    = True,
        max_length            = max_length,
        padding               = "max_length",
        truncation            = True,
        return_attention_mask = True,
        return_tensors        = "pt",
    )
    if labels is not None:
        encoding["labels"] = torch.tensor(labels, dtype=torch.long)
    return encoding


# ── Demo: batch of 4 texts ────────────────────────────────────────────────────
sample_texts = [
    "AI is transforming the world rapidly.",
    "BERT and XLM-RoBERTa are powerful for NLU tasks.",
    "Sports news: the championship final was incredible.",
    "A new vaccine trial shows promising early results.",
]
sample_labels = [0, 0, 1, 2]   # e.g. 0=tech, 1=sports, 2=health

bert_batch = prepare_bert_input(sample_texts, bert_tokenizer, max_length=24,
                                labels=sample_labels)
xlmr_batch = prepare_xlmr_input(sample_texts, xlmr_tokenizer, max_length=24,
                                 labels=sample_labels)

print("BERT batch tensors:")
for key, val in bert_batch.items():
    print(f"  {key:<22} shape: {tuple(val.shape)}  dtype: {val.dtype}")

print(f"\nXLM-RoBERTa batch tensors:")
for key, val in xlmr_batch.items():
    print(f"  {key:<22} shape: {tuple(val.shape)}  dtype: {val.dtype}")

# ── Attention mask visualization ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
for ax, (title, batch) in zip(axes, [("BERT", bert_batch), ("XLM-RoBERTa", xlmr_batch)]):
    mask = batch["attention_mask"].numpy()
    sns.heatmap(mask, ax=ax, cmap="Blues", vmin=0, vmax=1,
                xticklabels=False, yticklabels=[f"ex {i}" for i in range(len(sample_texts))],
                cbar=True, linewidths=0.3)
    ax.set_title(f"{title} — attention_mask\n(blue=real token, white=padding)", fontweight="bold")
    ax.set_xlabel("Token position")

plt.tight_layout()
plt.savefig("attention_masks.png", dpi=120, bbox_inches="tight")
plt.show()
print("Plot saved ✓")

## Part 4 — Loading and Exploring the Dataset

We use the **AG News** dataset — a 4-class news topic classification benchmark:
- `0` World  · `1` Sports  · `2` Business  · `3` Sci/Tech

It is loaded from Hugging Face, exported as `train.csv` / `test.csv`, then reloaded with `pd.read_csv` to match the exercise workflow.

In [ ]:
from datasets import load_dataset

# Load AG News from Hugging Face and export as CSV files
print("Loading AG News dataset…")
ag = load_dataset("ag_news")

# Export to CSV so we can demonstrate pd.read_csv()
ag["train"].to_csv("train.csv", index=False)
ag["test"].to_csv("test.csv",  index=False)
print("Exported  →  train.csv  /  test.csv")

# ── Load from CSV ─────────────────────────────────────────────────────────────
df_train = pd.read_csv("train.csv")
df_test  = pd.read_csv("test.csv")

LABEL_MAP = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}

# Map integer labels to names
df_train["label_name"] = df_train["label"].map(LABEL_MAP)
df_test["label_name"]  = df_test["label"].map(LABEL_MAP)

print(f"\n{'─'*50}")
print(f"df_train shape : {df_train.shape}")
print(f"df_test  shape : {df_test.shape}")
print(f"Columns        : {list(df_train.columns)}")

print(f"\n── First 5 rows of training set ──")
df_train[["text", "label", "label_name"]].head()

In [ ]:
# ── Dataset exploration ───────────────────────────────────────────────────────
print("── df_train.head() ──")
print(df_train[["text", "label", "label_name"]].head().to_string(index=False))

print(f"\n── df_train.shape : {df_train.shape}")
print(f"── df_test.shape  : {df_test.shape}")

print("\n── Label distribution (train) ──")
counts = df_train["label_name"].value_counts().sort_index()
for name, n in counts.items():
    print(f"  {name:<12} {n:>6,}  ({100*n/len(df_train):.1f}%)")

print("\n── Text length statistics ──")
df_train["text_len"] = df_train["text"].str.split().str.len()
print(df_train["text_len"].describe().round(1).to_string())

# ── Visualizations ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
COLORS = ["#4e91d4", "#e07b39", "#5cb85c", "#9b59b6"]

# Class distribution
label_counts = df_train["label_name"].value_counts().sort_index()
axes[0].bar(label_counts.index, label_counts.values, color=COLORS, edgecolor="white")
axes[0].set_title("Class Distribution — AG News (train)", fontweight="bold")
axes[0].set_ylabel("Number of examples")
axes[0].grid(True, alpha=0.3, axis="y")
for i, (lbl, cnt) in enumerate(label_counts.items()):
    axes[0].text(i, cnt + 80, f"{cnt:,}", ha="center", fontsize=9)

# Text length histogram per class
for i, (label_id, label_name) in enumerate(LABEL_MAP.items()):
    subset = df_train[df_train["label"] == label_id]["text_len"]
    axes[1].hist(subset, bins=40, alpha=0.5, color=COLORS[i], label=label_name)
axes[1].set_title("Text Length Distribution by Class", fontweight="bold")
axes[1].set_xlabel("Word count"); axes[1].set_ylabel("Frequency")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("dataset_exploration.png", dpi=120, bbox_inches="tight")
plt.show()
print("Plot saved ✓")

# Check for nulls
print(f"\nMissing values in train : {df_train.isnull().sum().to_dict()}")

## Part 5 — Creating Cross-Validation Folds

**StratifiedKFold** splits data into k folds while ensuring each fold has the same class proportion as the full dataset. This prevents a fold from being dominated by one class, giving fairer evaluation across all splits.

With **k=5**, each fold uses 80% of data for training and 20% for validation, and every example appears in the validation set exactly once.

In [ ]:
N_FOLDS = 5

# Work on a representative 10k subset so this is fast to demo
df_sub = df_train.sample(n=10_000, random_state=42).reset_index(drop=True)
texts  = df_sub["text"].tolist()
labels = df_sub["label"].tolist()

kf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

train_folds = []
val_folds   = []

print(f"StratifiedKFold  n_splits={N_FOLDS}  shuffle=True")
print(f"\n{'─'*70}")
print(f"{'Fold':<6} {'Train':>8} {'Val':>8}   Label distribution in val fold")
print(f"{'─'*70}")

for fold_idx, (train_idx, val_idx) in enumerate(kf.split(texts, labels)):
    train_folds.append({"texts":  [texts[i]  for i in train_idx],
                         "labels": [labels[i] for i in train_idx]})
    val_folds.append({"texts":    [texts[i]  for i in val_idx],
                       "labels":  [labels[i] for i in val_idx]})

    val_labels_fold = [labels[i] for i in val_idx]
    dist = {LABEL_MAP[k]: v for k, v in sorted(Counter(val_labels_fold).items())}
    dist_str = "  ".join(f"{n}:{c}" for n, c in dist.items())
    print(f"  {fold_idx+1:<4} {len(train_idx):>8,} {len(val_idx):>8,}   {dist_str}")

print(f"{'─'*70}")
print(f"\n✓ {N_FOLDS} folds created")
print(f"  Each training fold   : {len(train_folds[0]['texts']):,} examples")
print(f"  Each validation fold : {len(val_folds[0]['texts']):,} examples")

In [ ]:
# ── Visualise class balance across folds ──────────────────────────────────────
fig, axes = plt.subplots(1, N_FOLDS, figsize=(14, 4), sharey=True)

for fold_idx, (ax, val_fold) in enumerate(zip(axes, val_folds)):
    counts_f = Counter(val_fold["labels"])
    names_f  = [LABEL_MAP[k] for k in sorted(counts_f)]
    cnts_f   = [counts_f[k]  for k in sorted(counts_f)]
    ax.bar(names_f, cnts_f, color=COLORS, edgecolor="white")
    ax.set_title(f"Fold {fold_idx+1}", fontweight="bold")
    ax.set_ylim(0, max(cnts_f) * 1.25)
    ax.tick_params(axis="x", rotation=45, labelsize=8)
    ax.grid(True, alpha=0.3, axis="y")
    for j, c in enumerate(cnts_f):
        ax.text(j, c + 4, str(c), ha="center", fontsize=8)
    if fold_idx == 0:
        ax.set_ylabel("Validation examples")

plt.suptitle("Stratified K-Fold — Val Label Distribution per Fold",
             fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("kfold_class_distribution.png", dpi=120, bbox_inches="tight")
plt.show()
print("Plot saved ✓")

# ── Tokenize fold 0 as an example ────────────────────────────────────────────
print("\n── Tokenizing fold 0 training split with BERT (first 5 examples) ──")
fold0_train_batch = prepare_bert_input(
    train_folds[0]["texts"][:5],
    bert_tokenizer,
    max_length=128,
    labels=train_folds[0]["labels"][:5],
)
for key, val in fold0_train_batch.items():
    print(f"  {key:<22} shape: {tuple(val.shape)}  dtype: {val.dtype}")
print("\n✓ Fold 0 tokenization complete — ready for model training")

## Summary & Key Takeaways

| Part | Concept | Key point |
|---|---|---|
| 1 | BERT vs XLM-RoBERTa | BERT: WordPiece + `[CLS]/[SEP]` + NSP; XLM-R: SentencePiece + `<s></s>` + 100 languages, no NSP |
| 2 | `encode_plus` | Returns `input_ids`, `attention_mask`, `token_type_ids`; `decode()` reconstructs readable text |
| 3 | Input preparation | `padding="max_length"` + `truncation=True` → uniform tensor shape; `attention_mask=0` on pads |
| 4 | Dataset loading | `pd.read_csv()` → explore `.shape`, `.head()`, label counts, text length stats |
| 5 | StratifiedKFold | Preserves class ratios across all k folds; each example used for validation exactly once |

**Full fine-tuning workflow recap:**
```
CSV → pd.read_csv()
    → StratifiedKFold (5 splits)
        → for each fold:
            tokenizer.encode_plus() / prepare_bert_input()
            → BERT / XLM-RoBERTa + classifier head
            → AdamW, CrossEntropyLoss, 3–5 epochs
            → evaluate on val fold
    → average val metrics across 5 folds
```